# Tag-ID recognizer: train and evaluate on a GPU

Trains `TagNet` (two 36-way heads, A-Z0-9) on synthetic ear-tag crops, then measures exact-ID
accuracy versus character height in pixels. See [README.md](README.md) for the reasoning.

**How to use**
1. `git pull` on the GPU machine, open this notebook from `prototyping/tag-ocr/`.
2. Check the settings cell, then *Run All*. First run: ~15-45 min to render data (CPU), ~10-20 min to train.
3. Commit this notebook **with its outputs** plus `results/<RUN_NAME>/` (small text/PNG files).
   Weights and the image cache stay in the git-ignored `artifacts/` folder; do not commit them.

Results here are on synthetic data only. They do not show real-photo accuracy.

In [ ]:
# ---- Settings -------------------------------------------------------------
RUN_NAME = "run1"     # results go to artifacts/tag-ocr/<RUN_NAME> and results/<RUN_NAME>
N_TRAIN  = 200_000    # synthetic training crops (cached after the first render)
N_VAL    = 5_000
EPOCHS   = 30
BATCH    = 512        # drop to 256 if you hit CUDA out-of-memory
N_EVAL   = 1_000      # test crops per slice per pixel height
WORKERS  = None       # CPU render processes; None = cores - 2
SMOKE    = False      # True = tiny sizes, just to check everything runs
if SMOKE:
    N_TRAIN, N_VAL, EPOCHS, BATCH, N_EVAL = 2_000, 200, 1, 128, 20

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

HERE = Path.cwd()
assert (HERE / "train.py").is_file(), "Open this notebook from prototyping/tag-ocr/ (cwd must contain train.py)"
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))   # so `import synth` works in any kernel
REPO = HERE.parents[1]
RUN_DIR = REPO / "artifacts" / "tag-ocr" / RUN_NAME          # git-ignored: weights, cache
RESULTS = HERE / "results" / RUN_NAME                         # small files to commit
RESULTS.mkdir(parents=True, exist_ok=True)

def run(*args):
    """Run a script with this kernel's Python and stream its output."""
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=HERE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"{args[0]} exited with code {p.returncode}")

def opt(flag, value):
    return [flag, value] if value is not None else []

## 1. Environment check

In [ ]:
import torch, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA build", torch.version.cuda)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(), "| compute capability", cap,
          "| VRAM %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    arch = f"sm_{cap[0]}{cap[1]}"
    if arch not in torch.cuda.get_arch_list():
        print(f"WARNING: this torch build has no {arch} kernels. RTX 50-series needs a CUDA 12.8+ build:\n"
              "  pip install --force-reinstall torch --index-url https://download.pytorch.org/whl/cu128")
else:
    print("WARNING: no CUDA GPU visible; training will run on CPU and be very slow.")

import synth
for name in synth.FONT_SETS:
    try:
        print(f"fonts[{name}]:", [Path(f).name for f in synth.find_fonts(name)])
    except FileNotFoundError as e:
        print(f"fonts[{name}]: MISSING -", e)

In [ ]:
# Missing packages? Uncomment and run once, then restart the kernel.
# %pip install numpy opencv-python-headless pillow matplotlib
# Linux fonts (Liberation Serif stands in for Times New Roman):
#   sudo apt install fonts-liberation fonts-dejavu fonts-freefont-ttf

## 2. Look at the synthetic data

In [ ]:
from IPython.display import Image, Markdown, display
run("synth.py", "--out", RUN_DIR / "preview", "--n", 48, "--px", 6, 24, "--sheet")
shutil.copy(RUN_DIR / "preview" / "_sheet.png", RESULTS / "synthetic_preview.png")
display(Image(filename=str(RESULTS / "synthetic_preview.png")))

## 3. Train

In [ ]:
run("train.py", "--out", RUN_DIR, "--n-train", N_TRAIN, "--n-val", N_VAL,
    "--epochs", EPOCHS, "--bs", BATCH, *opt("--workers", WORKERS))

In [ ]:
import csv
import matplotlib
import matplotlib.pyplot as plt
rows = list(csv.DictReader(open(RUN_DIR / "train_log.csv")))
ep = [int(r["epoch"]) for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ep, [float(r["train_loss"]) for r in rows]); ax[0].set(title="train loss", xlabel="epoch")
ax[1].plot(ep, [float(r["val_exact"]) for r in rows], label="exact ID")
ax[1].plot(ep, [float(r["val_char"]) for r in rows], label="per character")
ax[1].set(title="validation accuracy (synthetic, 4-28 px)", xlabel="epoch", ylim=(0, 1)); ax[1].legend()
fig.tight_layout(); fig.savefig(RESULTS / "training_curves.png", dpi=110); plt.show()

## 4. Evaluate: accuracy vs character height

In [ ]:
run("evaluate.py", "--model", RUN_DIR / "tagnet.pt", "--n", N_EVAL, *opt("--workers", WORKERS))

In [ ]:
ev = list(csv.DictReader(open(RUN_DIR / "eval.csv")))
fig, ax = plt.subplots(figsize=(7, 4))
for s in dict.fromkeys(r["slice"] for r in ev):
    pts = [(float(r["char_px"]), float(r["exact_acc"])) for r in ev if r["slice"] == s]
    ax.plot(*zip(*pts), marker="o", label=s)
ax.axhline(0.95, ls="--", c="grey", lw=0.8)
ax.set(xlabel="character height (px)", ylabel="exact 2-char ID accuracy", ylim=(0, 1.01),
       title="TagNet on synthetic tags"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(RESULTS / "accuracy_vs_px.png", dpi=110); plt.show()
display(Markdown((RUN_DIR / "eval.md").read_text(encoding="utf-8")))

## 5. Save the small results for git

In [ ]:
for f in ["eval.md", "eval.csv", "train_log.csv"]:
    shutil.copy(RUN_DIR / f, RESULTS / f)
cfg = json.loads((RUN_DIR / "config.json").read_text())
cfg["train_fonts"] = [Path(f).name for f in cfg["train_fonts"]]   # drop machine-specific paths
cfg["out"] = f"artifacts/tag-ocr/{RUN_NAME}"
cfg["smoke"] = SMOKE
(RESULTS / "config.json").write_text(json.dumps(cfg, indent=2))
print("commit these:", sorted(p.name for p in RESULTS.iterdir()))
print("weights (do NOT commit):", RUN_DIR / "tagnet.pt")

## Reading the result

- Find the smallest pixel height where `times` accuracy reaches ~95%. Plug it into
  `python geometry.py` to get the maximum camera range for a given camera.
- A large gap between `times` and `heldout` means the model leans on the font; worth knowing
  before organizers confirm the font (RULE-003).
- `sel` columns: accuracy when the model is confident (>= 0.9). High `sel` with lower coverage
  means confidence-weighted voting across frames should work.
- Next: test on real photos of printed tags, which is the number that actually matters.